## <u><center><span style ="color:gold">Gradio</span> UI for <span style="color:orange">ChatGPT</span> Optimizer</u></center>
#### Per Shaw's Instructional Video ([here](https://www.youtube.com/watch?v=R5WXaxmb6m4)): there's no point in not taking this optimizer to the next level, giving it a UI for easier interaction
#### <u><span style="color:red">For this, we'll be using Python's library,</span> <span style ="color:gold">Gradio</span></u>.
##### * Gradio is a Python library that helps creat interactive user interfaces (UI's) for ML models and Python functions.
##### * With it, you can create a web application pretty easily, without deep-diving into more extensive WebDev frameworks.

In [1]:
# import "gradio" for UI
import gradio as gr

#import "requests" to simplify HTTP requests
import requests

# import functions, a file created in the job folder that houses the functions Gradio needs to do its job
from new_functions import (
    sanitize_input,
    prompt_creator,
    get_resume_response,
    process_resume,
    export_resume,
    cover_letter_prompt_creator,
    get_cover_response,
    save_cover_letter,
    is_posting_recent,
    template_detector,
    mentioned_on_socials,
    source_link_meta_date,
    detect_urgency_language,
    detect_job_board_source,
    career_page_search_url
)

# import BeautifulSoup to check for published jobdate is in posting's metadata - checking for link's age here.
from bs4 import BeautifulSoup 

#### Gradio is great because as you code from top-to-bottom, you are building your WebApp's appearance from top-to-bottom.
##### So as you code, you can visualize where the code is going and easily adjust things to your liking later.

In [2]:
with gr.Blocks(css="""
    .section-header {
        text-align: center;
        color: #1e90ff;
        font-size: 2.5em;
        font-family: 'Segoe UI', sans-serif;
        font-weight: bold;
        margin-top: 20px;
        margin-bottom: 10px;
    }
    .subtext {
        text-align: center;
        font-size: 1.1em;
        color: #555;
        margin-bottom: 30px;
    }
    .gr-button {
        font-weight: bold;
        font-size: 1.1em;
        padding: 0.75em 1.5em;
        border-radius: 12px;
    }
""") as app:

    # Header
    gr.Markdown("<div class='section-header'>🚀 Your One-Stop Resume Shop</div>")
    gr.Markdown("""
        <div class='subtext'>
            Upload your resume, enter the job info, and instantly generate a tailored resume and compelling cover letter.  
            You can also validate whether the job is real, relevant, and recent.
        </div>
    """)

    # Inputs
    with gr.Row():
        resume_input = gr.File(label="📄 Upload Markdown Resume")
        company_input = gr.Textbox(label="🏢 Company Name", placeholder="e.g., Data Clymer")
        job_input = gr.Textbox(label="📝 Job Description", lines=8, placeholder="Paste full job description here")

    with gr.Tab("✨ Resume Optimizer"):
        run_resume = gr.Button("✨ Optimize Resume")
        resume_md = gr.Markdown(label="Optimized Resume (Markdown)")
        resume_edit = gr.Textbox(label="✏️ Edit Your Resume Below", lines=10, interactive=True)
        suggestions = gr.Markdown(label="🔍 Suggestions")
        export_resume_btn = gr.Button("⬇ Export Resume as PDF")
        export_resume_result = gr.Markdown()

    with gr.Tab("📬 Cover Letter Generator"):
        run_cover = gr.Button("📝 Click Here To Generate Your Cover Letter")
        cover_output = gr.Textbox(label="Generated Cover Letter - Please Proofread and Check for <<cough cough>> Accuracy. 'I cured Polio!' No, you didn't.", lines=15, interactive=True)
        export_cover_btn = gr.Button("⬇ Export Cover Letter as PDF")
        export_cover_result = gr.Markdown()
        
    with gr.Tab("🔍 Job Validator"):
        gr.Markdown("<div class='section-header'>🕵️‍♂️ Job Validation Tool</div>")
        gr.Markdown("<div class='subtext'>Is this a real job, or just an attempt to mine your data? Check how recent, relevant, and visible a job post is on socials.</div>")

        with gr.Row():
            job_url = gr.Textbox(label="🔗 Optional: Job Posting URL", placeholder="https://jobs.greenhouse.io/...")
            jd_date = gr.Textbox(label="📅 Job Posting Date (Format: YYYY-MM-DD)", placeholder="e.g., 2025-06-01, or your best guess.")
            job_title = gr.Textbox(label="🔖 Job Title", placeholder="e.g., Technical Support Analyst")

        jd_validate_btn = gr.Button("✅ Run Job Validation")
        jd_validation_result = gr.Markdown()

        def source_link_meta_date(url):
            try:
                resp = requests.get(url, timeout=5)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Look for known meta tags
                for tag in ["datePublished", "article:published_time", "og:published_time"]:
                    meta = soup.find("meta", {"property": tag}) or soup.find("meta", {"name": tag})
                    if meta and meta.get("content"):
                        return meta["content"]
            except Exception as e:
                return None
            return None

        def validate_job(posting_date, company, job_title, job_description, job_url):
            # handle user error / lack of info
            if not company or not job_description:
                return "Please enter at least the company name and job description."
                
            # Existing checks
            recent = is_posting_recent(posting_date)
            template_flag = template_detector(job_description)
            urgency_flag = detect_urgency_language(job_description)

            # New ones
            meta_date = source_link_meta_date(job_url)
            board_check = detect_job_board_source(job_url)
            career_search = career_page_search_url(company, job_title)

            report = f"### 🕒 Posting Date:\n"
            report += "✅ User-entered date is recent.\n" if recent else "⚠️ Possibly outdated.\n"
            report += f"📅 Meta date from page: `{meta_date}`\n" if meta_date else "📅 No meta publish date found.\n"

            report += f"\n### 🤖 Template Detection:\n"
            report += "⚠️ Generic language detected.\n" if template_flag else "✅ Description looks specific.\n"

            report += f"\n### 🚨 Urgency Signal:\n"
            report += "✅ Urgency signals found in description.\n" if urgency_flag else "⚠️ No urgency keywords found.\n"

            report += f"\n### 🌐 Job Source:\n{board_check}\n"

            report += f"\n### 🧭 Career Site Check:\nTry searching: [Google Career Match]({career_search})"

            return report
    
    jd_validate_btn.click(
        fn=validate_job,
        inputs=[jd_date, company_input, job_title, job_input, job_url],
        outputs=jd_validation_result
    )

    # Resume events
    run_resume.click(fn=process_resume, inputs=[resume_input, job_input], outputs=[resume_md, resume_edit, suggestions])
    export_resume_btn.click(fn=export_resume, inputs=[resume_edit, company_input], outputs=[export_resume_result])

    # Cover letter events
    def generate_cover_letter(resume_file, job_desc):
        with open(resume_file.name, "r", encoding="utf-8") as f:
            resume_txt = f.read()
        prompt = cover_letter_prompt_creator(resume_txt, job_desc)
        return get_cover_response(prompt)

    def export_cover_handler(cover_text, company):
        pdf, md = save_cover_letter(cover_text, company)
        return f"✅ PDF saved at: `{pdf}`" if pdf else "❌ Failed to export."

    run_cover.click(fn=generate_cover_letter, inputs=[resume_input, job_input], outputs=[cover_output])
    export_cover_btn.click(fn=export_cover_handler, inputs=[cover_output, company_input], outputs=[export_cover_result])

# Launch
app.launch(server_name="0.0.0.0", server_port=8080)

* Running on local URL:  http://0.0.0.0:8080

To create a public link, set `share=True` in `launch()`.


In [ ]:
# # construct and functionalize the Gradio UI
# with gr.Blocks() as app:
#     # Header
#     gr.Markdown("Your One-Stop Resume and Cover Letter Optimizer")
#     gr.Markdown("""
#     1. Drop your Markdown resume below.  
#     2. Enter the company name.  
#     3. Paste the job description.  
#     4. Generate both your tailored resume **and** a compelling cover letter in Markdown & PDF format.
#     """)

#     # Inputs
#     with gr.Row():
#         resume_input = gr.File(label="Drop Your Markdown Resume Here")
#         company_input = gr.Textbox(label="The Company with Whom You're Applying", placeholder="e.g., Data Clymer")
#         job_input = gr.Textbox(label="Paste In the Job Description Here", lines=10, interactive=True)

#     with gr.Tab("Resume Optimizer"):
#         run_resume = gr.Button("✨ Optimize Resume")
#         resume_md = gr.Markdown(label="Optimized Resume (Markdown View)")
#         resume_edit = gr.Textbox(label="Edit Your Resume Below", lines=10, interactive=True)
#         suggestions = gr.Markdown(label="Suggestions")
#         export_resume_btn = gr.Button("⬇ Export Resume as PDF")
#         export_resume_result = gr.Markdown()

#     with gr.Tab("Cover Letter Generator"):
#         run_cover = gr.Button("📝 Generate Cover Letter")
#         cover_output = gr.Textbox(label="Generated Cover Letter (Markdown)", lines=15, interactive=True)
#         export_cover_btn = gr.Button("⬇ Export Cover Letter as PDF")
#         export_cover_result = gr.Markdown()
        
#     with gr.Tab("Job Validator"):
#         gr.Markdown("Validate whether a job post is recent, specific, and visible on social platforms.")

#         with gr.Row():
#             jd_date = gr.Textbox(label="Job Posting Date (YYYY-MM-DD)", placeholder="e.g., 2025-06-01")
#             jd_company = gr.Textbox(label="Company Name", placeholder="e.g., IEM, LLC")

#         jd_title = gr.Textbox(label="Job Title", placeholder="e.g., Senior Data Engineer")
#         jd_desc = gr.Textbox(label="Paste Full Job Description", lines=10)

#         jd_validate_btn = gr.Button("✅ Run Job Validation")
#         jd_validation_result = gr.Markdown()

#         # Validation click logic
#         def validate_job(posting_date, company, job_title, job_description):
#             recent = is_posting_recent(posting_date)
#             template_flag = template_detector(job_description)
#             social_links = mentioned_on_socials(company, job_title)

#             report = f"### 🕒 Posting Date Check:\n"
#             report += "✅ Job appears to be recent.\n" if recent else "⚠️ Job may be outdated (posted over 60 days ago).\n"

#             report += f"\n### 🤖 Template Language Detection:\n"
#             report += "⚠️ This job description uses generic/template language.\n" if template_flag else "✅ Description looks specific.\n"

#             report += f"\n### 🔍 Social Media Mentions:\n"
#             report += f"- [Search on X](<{social_links['x']}>)\n"
#             report += f"- [Search on LinkedIn](<{social_links['linkedin']}>)\n"

#             return report

#         jd_validate_btn.click(
#             fn=validate_job,
#             inputs=[jd_date, jd_company, jd_title, jd_desc],
#             outputs=jd_validation_result
#         )

#     # Resume events
#     run_resume.click(fn=process_resume, inputs=[resume_input, job_input], outputs=[resume_md, resume_edit, suggestions])
#     export_resume_btn.click(fn=export_resume, inputs=[resume_edit, company_input], outputs=[export_resume_result])

#     # Cover letter events
#     def generate_cover_letter(resume_file, job_desc):
#         with open(resume_file.name, "r", encoding="utf-8") as f:
#             resume_txt = f.read()
#         prompt = cover_letter_prompt_creator(resume_txt, job_desc)
#         return get_cover_response(prompt)

#     def export_cover_handler(cover_text, company):
#         pdf, md = save_cover_letter(cover_text, company)
#         return f"✅ PDF saved at: `{pdf}`" if pdf else "❌ Failed to export."

#     run_cover.click(fn=generate_cover_letter, inputs=[resume_input, job_input], outputs=[cover_output])
#     export_cover_btn.click(fn=export_cover_handler, inputs=[cover_output, company_input], outputs=[export_cover_result])

# # Launch
# app.launch(server_name="0.0.0.0", server_port=8080)

